# CDCR Top 10 Prisons by Hazard Indicator

Top 10 prisons ranked independently for each of 6 hazard indicators, plus raw index columns for sorting.

**Hazards:** Days over 80°F (mid-century), SPEI-12 drought frequency, drought index, % years very wet, flood index, fire hazard severity zone.  
**Cell format:** Name (cdcr_code) / City, CA / (stat)  
**Excludes:** CRC (closing 2026), CAC, FWF, CVSP

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from pathlib import Path

ROOT = Path('../../')
DATA = ROOT / 'data'
DATA_SRC = ROOT / 'data_sources'

In [2]:
# Load data sources
cdcr = pd.read_csv(DATA / 'cdcr' / 'cdcr_facilities.csv')
all_fac = pd.read_csv(DATA / 'allfacilities_climate_hazards.csv')
heat_tracts = pd.read_csv(DATA_SRC / 'hazards' / 'heat' / 'heatdays_alltimes_tract.csv')
drought_tracts = pd.read_csv(DATA_SRC / 'hazards' / 'drought' / 'droughtfrequency_tract.csv')
flood_tracts = pd.read_csv(DATA / 'hazards' / 'flood_hazard.csv')

# Filter to 30 active state prisons
cdcr_active = cdcr[
    (cdcr['cdcr_code'].notna()) &
    (cdcr['cdcr_code'].str.strip() != '') &
    (cdcr['cdcr_code'] != 'CRC') &
    (~cdcr['cdcr_code'].isin(['CAC', 'FWF', 'CVSP']))
].copy()

print(f'Active CDCR state prisons: {len(cdcr_active)}')
assert len(cdcr_active) == 30

Active CDCR state prisons: 30


In [3]:
# Build base dataframe with facility info + hazard indices from master file
df = cdcr_active[['cdcr_code', 'name', 'city', 'facilityid', 'tract_geoid', 'latitude', 'longitude']].merge(
    all_fac[['facilityid',
             'drought_hazard_historic_idx', 'drought_hazard_midcentury_idx',
             'flood_hazard_historic_idx', 'flood_hazard_midcentury_idx']],
    on='facilityid', how='left'
)

# Normalize tract GEOIDs to 11-digit strings
df['tract_geoid'] = df['tract_geoid'].astype(str).str.zfill(11)
heat_tracts['GEOID'] = heat_tracts['GEOID'].astype(str).str.zfill(11)
drought_tracts['GEOID'] = drought_tracts['GEOID'].astype(str).str.zfill(11)
flood_tracts['GEOID'] = flood_tracts['GEOID'].astype(str).str.zfill(11)

# Join heat 80F data from tract file
df = df.merge(
    heat_tracts[['GEOID', 'days_over_80_historic', 'days_over_80_midcentury']],
    left_on='tract_geoid', right_on='GEOID', how='left'
).drop(columns='GEOID')

# Join drought frequency from tract file
df = df.merge(
    drought_tracts[['GEOID', 'drought_freq_spei12_midcentury', 'drought_delta_spei12_midcentury']],
    left_on='tract_geoid', right_on='GEOID', how='left'
).drop(columns='GEOID')

# Join wet years from flood tract file
df = df.merge(
    flood_tracts[['GEOID', 'flood_verywet_pre_pct', 'flood_verywet_fut_pct']],
    left_on='tract_geoid', right_on='GEOID', how='left'
).drop(columns='GEOID')

# ── Fire hazard: two-pass spatial join with CalFire FHSZ ──────────────────
# Pass 1: strict point-in-polygon (no buffer) for facilities inside a zone
# Pass 2: 250m buffer for unclassified facilities only (never upgrades pass 1)
BUFFER_M = 250
severity_order = {'Very High': 3, 'High': 2, 'Moderate': 1, 'NonWildland': 0}

fhsz = gpd.read_file(DATA_SRC / 'hazards' / 'wildfire' / 'calfire_fhsz.geojson')
if fhsz.crs is None:
    fhsz = fhsz.set_crs('EPSG:4326')
fhsz_proj = fhsz.to_crs('EPSG:3310')
fhsz_cols = fhsz_proj[['fhsz', 'responsibility', 'geometry']]

# Build points in projected CRS
prison_pts = gpd.GeoDataFrame(
    df[['cdcr_code']].copy(),
    geometry=[Point(xy) for xy in zip(df['longitude'], df['latitude'])],
    crs='EPSG:4326'
).to_crs('EPSG:3310')

# Pass 1: strict within
pass1 = gpd.sjoin(prison_pts.copy(), fhsz_cols, how='left', predicate='within')
pass1['_sev'] = pass1['fhsz'].map(severity_order).fillna(-1)
pass1 = pass1.sort_values('_sev', ascending=False).drop_duplicates('cdcr_code')
pass1.loc[pass1['fhsz'] == 'NonWildland', 'fhsz'] = np.nan

# Pass 2: buffered intersects, only for facilities unclassified in pass 1
unclassified = pass1[pass1['fhsz'].isna()]['cdcr_code'].tolist()
if unclassified:
    pts_buf = prison_pts[prison_pts['cdcr_code'].isin(unclassified)].copy()
    pts_buf['geometry'] = pts_buf.geometry.buffer(BUFFER_M)
    pass2 = gpd.sjoin(pts_buf, fhsz_cols, how='left', predicate='intersects')
    pass2['_sev'] = pass2['fhsz'].map(severity_order).fillna(-1)
    pass2 = pass2.sort_values('_sev', ascending=False).drop_duplicates('cdcr_code')
    pass2.loc[pass2['fhsz'] == 'NonWildland', 'fhsz'] = np.nan

    # Merge pass 2 results into pass 1
    for _, row in pass2[pass2['fhsz'].notna()].iterrows():
        pass1.loc[pass1['cdcr_code'] == row['cdcr_code'], 'fhsz'] = row['fhsz']
        pass1.loc[pass1['cdcr_code'] == row['cdcr_code'], 'responsibility'] = row['responsibility']

result = pass1.rename(columns={'fhsz': 'fire_fhsz', 'responsibility': 'fire_responsibility'})
df = df.merge(result[['cdcr_code', 'fire_fhsz', 'fire_responsibility']], on='cdcr_code', how='left')

print(f'Fire FHSZ (strict within + {BUFFER_M}m buffer for unclassified):')
print(df[['cdcr_code', 'fire_fhsz', 'fire_responsibility']].sort_values('fire_fhsz', na_position='last').to_string(index=False))
print(f'\nJoined rows: {len(df)}')
print(f'Classified: {df["fire_fhsz"].notna().sum()}, Unclassified: {df["fire_fhsz"].isna().sum()}')

Fire FHSZ (strict within + 250m buffer for unclassified):
cdcr_code fire_fhsz fire_responsibility
     MCSP      High                 LRA
      CIW      High                 LRA
      CAL  Moderate                 LRA
      SOL  Moderate                 LRA
      CMC  Moderate                 SRA
     PBSP  Moderate                 LRA
     HDSP  Moderate                 LRA
      FOL  Moderate                 LRA
      CCI  Moderate                 LRA
      CTF  Moderate                 LRA
      SAC  Moderate                 LRA
      LAC  Moderate                 LRA
      ISP  Moderate                 LRA
     SVSP  Moderate                 LRA
      RJD Very High                 SRA
      SCC Very High                 SRA
      CEN       NaN                 LRA
      ASP       NaN                 LRA
      COR       NaN                 LRA
     SATF       NaN                 LRA
     CCWF       NaN                 LRA
     PVSP       NaN                 LRA
      VSP       NaN   

In [4]:
# Compute derived stats

# Heat: % increase from historic to mid-century (80F)
df['heat_80f_pct_change'] = ((df['days_over_80_midcentury'] - df['days_over_80_historic'])
                              / df['days_over_80_historic'] * 100)

# Wet years: % increase from historic to mid-century
df['verywet_pct_change'] = ((df['flood_verywet_fut_pct'] - df['flood_verywet_pre_pct'])
                             / df['flood_verywet_pre_pct'] * 100)

# Drought index: % increase from historic to mid-century
df['drought_idx_pct_change'] = ((df['drought_hazard_midcentury_idx'] - df['drought_hazard_historic_idx'])
                                 / df['drought_hazard_historic_idx'] * 100)

# Flood index: % increase from historic to mid-century
df['flood_idx_pct_change'] = ((df['flood_hazard_midcentury_idx'] - df['flood_hazard_historic_idx'])
                               / df['flood_hazard_historic_idx'] * 100)

# Encode FHSZ as ordinal for ranking
fhsz_ordinal = {'Very High': 3, 'High': 2, 'Moderate': 1}
df['fire_fhsz_ordinal'] = df['fire_fhsz'].map(fhsz_ordinal).fillna(0)

# FHSZ display label
label_map = {'Very High': 'Very High Hazard', 'High': 'High Hazard', 'Moderate': 'Moderate Hazard'}
df['fire_label'] = df['fire_fhsz'].map(label_map).fillna('No Hazard')

print('Stats computed.')
print(f'\nFHSZ distribution:')
print(df['fire_label'].value_counts())

Stats computed.

FHSZ distribution:
fire_label
No Hazard           14
Moderate Hazard     12
Very High Hazard     2
High Hazard          2
Name: count, dtype: int64


In [5]:
# Build formatted cell strings for each hazard

def fmt_cell(row, stat1_val, stat1_unit, stat2_val, stat2_unit):
    """Format: Name (CODE)\nCity, CA\n(stat1 unit1, stat2 unit2)"""
    header = f"{row['name']} ({row['cdcr_code']})"
    location = f"{row['city']}, CA"
    stat = f"({stat1_val} {stat1_unit}, {stat2_val} {stat2_unit})"
    return f"{header}\n{location}\n{stat}"

def fmt_cell_single(row, stat_val, stat_unit):
    """Format for single-stat cells (fire)"""
    header = f"{row['name']} ({row['cdcr_code']})"
    location = f"{row['city']}, CA"
    stat = f"({stat_val} {stat_unit})"
    return f"{header}\n{location}\n{stat}"

# Heat 80F
df['heat_cell'] = df.apply(lambda r: fmt_cell(
    r,
    f"{r['days_over_80_midcentury']:.0f}", 'days',
    f"+{r['heat_80f_pct_change']:.0f}%", 'increase'
), axis=1)

# Drought frequency
df['drought_freq_cell'] = df.apply(lambda r: fmt_cell(
    r,
    f"{r['drought_freq_spei12_midcentury']:.1f}%", 'of years',
    f"+{r['drought_delta_spei12_midcentury']:.0f}%", 'increase'
), axis=1)

# Drought index
df['drought_idx_cell'] = df.apply(lambda r: fmt_cell(
    r,
    f"{r['drought_hazard_midcentury_idx']:.1f}", 'index',
    f"+{r['drought_idx_pct_change']:.0f}%", 'increase'
), axis=1)

# Very wet years
df['verywet_cell'] = df.apply(lambda r: fmt_cell(
    r,
    f"{r['flood_verywet_fut_pct']:.1f}%", 'of years',
    f"+{r['verywet_pct_change']:.0f}%", 'increase'
), axis=1)

# Flood index
df['flood_idx_cell'] = df.apply(lambda r: fmt_cell(
    r,
    f"{r['flood_hazard_midcentury_idx']:.1f}", 'index',
    f"+{r['flood_idx_pct_change']:.0f}%", 'increase'
), axis=1)

# Fire FHSZ
df['fire_cell'] = df.apply(lambda r: fmt_cell_single(
    r, r['fire_label'], ''
).replace('  ', ' ').replace(' )', ')'), axis=1)

print('Cell formatting done.')
print('\nExample heat cell:')
print(df.sort_values('days_over_80_midcentury', ascending=False).iloc[0]['heat_cell'])

Cell formatting done.

Example heat cell:
Calipatria State Prison (CAL)
Calipatria, CA
(271 days, +12% increase)


In [6]:
# Build the top-10 table: each hazard column ranked independently

TOP_N = 10

# Rank by each hazard
heat_top = df.nlargest(TOP_N, 'days_over_80_midcentury', keep='first').reset_index(drop=True)

drought_freq_top = df.nlargest(TOP_N, 'drought_freq_spei12_midcentury', keep='first').reset_index(drop=True)

drought_idx_top = df.nlargest(TOP_N, 'drought_hazard_midcentury_idx', keep='first').reset_index(drop=True)

verywet_top = df.nlargest(TOP_N, 'flood_verywet_fut_pct', keep='first').reset_index(drop=True)

flood_idx_top = df.nlargest(TOP_N, 'flood_hazard_midcentury_idx', keep='first').reset_index(drop=True)

fire_top = df.nlargest(TOP_N, 'fire_fhsz_ordinal', keep='first').reset_index(drop=True)

# Assemble the table
table = pd.DataFrame({
    'Rank': range(1, TOP_N + 1),
    'Days Over 80°F\nMid-Century': heat_top['heat_cell'].values,
    'Drought Frequency\nSPEI-12 Mid-Century': drought_freq_top['drought_freq_cell'].values,
    'Drought Index\nMid-Century': drought_idx_top['drought_idx_cell'].values,
    'Very Wet Years\nMid-Century': verywet_top['verywet_cell'].values,
    'Flood Index\nMid-Century': flood_idx_top['flood_idx_cell'].values,
    'Fire Hazard\nSeverity Zone': fire_top['fire_cell'].values,
    'drought_idx_midcentury': drought_idx_top['drought_hazard_midcentury_idx'].round(1).values,
    'drought_idx_change_pct': drought_idx_top['drought_idx_pct_change'].round(0).astype(int).values,
    'flood_idx_midcentury': flood_idx_top['flood_hazard_midcentury_idx'].round(1).values,
    'flood_idx_change_pct': flood_idx_top['flood_idx_pct_change'].round(0).astype(int).values,
})

table = table.set_index('Rank')
table

,Days Over 80°F\nMid-Century,Drought Frequency\nSPEI-12 Mid-Century,Drought Index\nMid-Century,Very Wet Years\nMid-Century,Flood Index\nMid-Century,Fire Hazard\nSeverity Zone,drought_idx_midcentury,drought_idx_change_pct,flood_idx_midcentury,flood_idx_change_pct
Rank,,,,,,,,,,
1,"Calipatria State Prison (CAL)\nCalipatria, CA\...","Ironwood State Prison (ISP)\nBlythe, CA\n(32.2...",Central California Women'S Facility (CCWF)\nCh...,California Men'S Colony (CMC)\nSan Luis Obispo...,California State Prison-Los Angeles County (LA...,R J Donovan Correctional Facility (RJD)\nSan D...,78.4,-1,82.3,333
2,"Ironwood State Prison (ISP)\nBlythe, CA\n(260 ...","Calipatria State Prison (CAL)\nCalipatria, CA\...","Valley State Prison (VSP)\nChowchilla, CA\n(78...","Avenal State Prison (ASP)\nAvenal, CA\n(30.4% ...","California State Prison, Corcoran (COR)\nCorco...","Sierra Conservation Center (SCC)\nJamestown, C...",78.4,-1,38.7,2
3,"Centinela State Prison (CEN)\nImperial, CA\n(2...","Centinela State Prison (CEN)\nImperial, CA\n(3...",California Health Care Facility (CHCF)\nStockt...,California Institution For Women (CIW)\nCorona...,Ca Substance Abuse Treatment Facility (SATF)\n...,"Mule Creek State Prison (MCSP)\nIone, CA\n(Hig...",70.4,0,38.7,2
4,California Institution For Women (CIW)\nCorona...,California State Prison-Los Angeles County (LA...,"North Kern State Prison (NKSP)\nDelano, CA\n(6...","California State Prison, Solano (SOL)\nVacavil...","Pleasant Valley State Prison (PVSP)\nCoalinga,...",California Institution For Women (CIW)\nCorona...,66.1,1,34.3,20
5,"California Institution For Men (CIM)\nChino, C...",R J Donovan Correctional Facility (RJD)\nSan D...,"Kern Valley State Prison (KVSP)\nDelano, CA\n(...","California Medical Facility (CMF)\nVacaville, ...",California Institution For Women (CIW)\nCorona...,"Calipatria State Prison (CAL)\nCalipatria, CA\...",65.8,1,33.7,40
6,"Wasco State Prison (WSP)\nWasco, CA\n(200 days...","California Institution For Men (CIM)\nChino, C...","Mule Creek State Prison (MCSP)\nIone, CA\n(63....","Pleasant Valley State Prison (PVSP)\nCoalinga,...",California Men'S Colony (CMC)\nSan Luis Obispo...,"California State Prison, Solano (SOL)\nVacavil...",63.7,1,30.8,9
7,"Kern Valley State Prison (KVSP)\nDelano, CA\n(...",California Institution For Women (CIW)\nCorona...,"Sierra Conservation Center (SCC)\nJamestown, C...",Pelican Bay State Prison (PBSP)\nCrescent City...,Pelican Bay State Prison (PBSP)\nCrescent City...,California Men'S Colony (CMC)\nSan Luis Obispo...,63.6,4,28.8,7
8,"California State Prison, Corcoran (COR)\nCorco...",California Correctional Institution (CCI)\nTeh...,"California State Prison, Corcoran (COR)\nCorco...","Correctional Training Facility (CTF)\nSoledad,...","Kern Valley State Prison (KVSP)\nDelano, CA\n(...",Pelican Bay State Prison (PBSP)\nCrescent City...,61.8,1,28.6,-1
9,Ca Substance Abuse Treatment Facility (SATF)\n...,"Wasco State Prison (WSP)\nWasco, CA\n(17.5% of...",Ca Substance Abuse Treatment Facility (SATF)\n...,"Salinas Valley State Prison (SVSP)\nSoledad, C...","Avenal State Prison (ASP)\nAvenal, CA\n(27.3 i...","High Desert State Prison (HDSP)\nSusanville, C...",61.8,1,27.3,3


In [7]:
# Save to CSV
table.to_csv('CDCR_hazard_top10_table.csv')
print('Saved CDCR_hazard_top10_table.csv')

Saved CDCR_hazard_top10_table.csv
